# Exercise - Hyperparameter Tuning with Grid Search

In this exercise you will train a base model and then try to find better combinations of hyperparameter values using the grid search technique.

In [ ]:
# DO NOT MODIFY - imports
import pandas as pd
import numpy as np

## 1. Setup

Execute the cells below to read prepared data on the [Invesco QQQ Trust, Series 1 (NASDAQ: QQQ)](https://finance.yahoo.com/quote/QQQ/) ETF from 1999 to 2017. We have already engineered some technical indicators as features and cleaned the data. The DataFrame also includes the raw level of the VIX (Volatility Index).

In [ ]:
# DO NOT MODIFY - load data and display basic statistics
df = pd.read_csv("Data.csv")
df.describe()

In [ ]:
df.tail()

We'd like to try and predict the direction of 5-day future returns. Run the cell below to split the data and prepare for model training.

In [ ]:
# DO NOT MODIFY - Define features and target and split data
from sklearn.model_selection import train_test_split

X = df.drop(columns=["fut_ret_5d_is_pos", "Date"])
y = df["fut_ret_5d_is_pos"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

## 2. Training a base model

Train a `RandomForestClassifier` and train it using its default hyperparameter values. As this is a tree-based model, you do not need to scale the features.

In [ ]:
# DO NOT MODIFY - imports
from sklearn.ensemble import RandomForestClassifier

# FILL IN - Instantiate a RandomForestClassifier and fit it to the training data
# Use random_state=52 for reproducibility
# Set n_jobs=-1 to enable parallel processing using all available CPU cores
clf = RandomForestClassifier(random_state=52, n_jobs=-1)
clf.fit(X_train, y_train)

We will focus on precision as our performance metric, as we would like to avoid False Positives as much as possible.  
Below, we have provided a function that plots 5-fold cross-validated precision scores. Study it and invoke it to plot the learning curves using the training set. You should be able to observe that the model is overfitting to the training set.

In [ ]:
# DO NOT MODIFY - imports
import matplotlib.pyplot as plt
from sklearn.model_selection import learning_curve


# DO NOT MODIFY - plotter function
def plot_learning_curve(model, X, y, cv=5, n_jobs=-1):
    train_sizes, train_scores, test_scores = learning_curve(
        model,
        X,
        y,
        cv=cv,
        n_jobs=n_jobs,
        scoring="precision",
    )
    train_scores_mean = np.mean(train_scores, axis=1)
    test_scores_mean = np.mean(test_scores, axis=1)
    plt.plot(train_sizes, train_scores_mean, label="CV training score")
    plt.plot(train_sizes, test_scores_mean, label="CV test score")
    plt.title("Learning curve for Random Forest Classifier")
    plt.xlabel("Training examples")
    plt.ylabel("Precision")
    plt.legend()
    plt.show()


# FILL IN - Plot the learning curve for the RandomForestClassifier using the training data
plot_learning_curve(clf, X_train, y_train)

What was the average cross-validated precision score on the training set?

In [ ]:
# DO NOT MODIFY - imports
from sklearn.model_selection import cross_val_score

# FILL IN - Get the 5-fold cross-validated precision of the classifer on the training data
# Use n_jobs=-1 for parallel processing
precision_training_data = cross_val_score(
    clf, X_train, y_train, cv=5, n_jobs=-1, scoring="precision"
)
print("Precision of the training data: ", precision_training_data.mean())

And how does this score compare to the precision on the test set? - **HINT:** Use the fitted classifier's `predict()` method to get an array of predictions on the test set.

In [ ]:
# DO NOT MODIFY - imports
from sklearn.metrics import precision_score

# FILL IN - Get the precision of the classifier on the test data
y_test_predictions = clf.predict(X_test)
precision = precision_score(y_test, y_test_predictions)
print("Precision of the classifier on the test data: ", precision)

## 3. Grid search

Recall that you can use the `get_params()` method of the classifier to see a list of its hyperparameter and other settings.

In [ ]:
clf.get_params()

Below, we have picked 3 different values for each of the 4 major hyperparameters of `RandomForestClassifier`. Using Scikit-Learn's `GridSearchCV` class, perform a 5-fold cross-validated grid search using the provided search grid.

In [ ]:
# DO NOT MODIFY - imports
from sklearn.model_selection import GridSearchCV

# DO NOT MODIFY - the `hyperspace` of hyperparameters to search
search_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [5, 10, 20],
    "min_samples_split": [2, 7, 15],
    "min_samples_leaf": [1, 2, 4],
}

# FILL IN - Instantiate a GridSearchCV object with the fitted RandomForestClassifier, the search grid, 5-fold cross-validation, and precision scoring.
# Fit it to the training data
# Don't forget to set n_jobs=-1 for parallel processing. This may take a minute or two even with parallel processing.
grid_search = GridSearchCV(
    clf, param_grid=search_grid, scoring="precision", cv=5, n_jobs=-1
)
grid_search.fit(X_train, y_train)

Store the best parameters, best score, and best estimator (model). (These are attributes of `search`.) Feel free to print out the best CV precision score. How does it compare to the base model? Which combination of values yielded this result?

In [ ]:
# FILL IN - Get the best parameters, best score, and best estimator from the GridSearchCV object
best_params = grid_search.best_params_
best_score = grid_search.best_score_
best_estimator = grid_search.best_estimator_

In [ ]:
best_score

In [ ]:
best_params

Run the cell below to see the top 5 results in detail.

In [ ]:
search_results = pd.DataFrame(grid_search.cv_results_)  # type: ignore
search_results.sort_values("rank_test_score").head()

Re-use the same `plot_learning_curve()` function we provided earlier to plot the learning curve for the best estimator found using training data. How does it compare to the learning curve of the original classifier?

In [ ]:
# FILL IN - Plot the learning curve for the best estimator using the training data
plot_learning_curve(best_estimator, X_train, y_train)

And finally, use this estimator to evaluate the test set and get the new test performance score. How does it compare?

In [ ]:
# FILL IN - Get the precision of the best model on test data
y_test_predictions = best_estimator.predict(X_test)
precision = precision_score(y_test, y_test_predictions)
print("Precision of the best classifier on the test data: ", precision)

In [ ]:
# FILL IN - Get predictions on the training and test sets
y_pred_train = best_estimator.predict(X_train)
y_pred_test = best_estimator.predict(X_test)

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns


def plot_confusion_matrix(y_true, y_pred):
    conf_mat = confusion_matrix(y_true, y_pred)
    sns.heatmap(conf_mat, annot=True, fmt="d", cmap="Blues")
    plt.xlabel("Predicted Labels")
    plt.ylabel("Actual Labels")
    plt.title("Confusion Matrix")
    plt.show()


plot_confusion_matrix(y_test, y_pred_test)